|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 2:</h2>|<h1>Batching<h1>|
|<h2>Section:</h2>|<h1>Static batching<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: pick the batch size that is actually fastest<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(2)

You have measured how much faster a big batch is. Now find the batch size
that is actually fastest, which is not the same question.

All simulation, no GPU. The throughput numbers come from the demo notebook in
this folder.

In [ ]:
### run this cell

lengths = rng.lognormal(mean=np.log(120), sigma=0.9, size=20000).astype(int) + 1

# The one hardware number in this notebook. B_ridge is the batch at which a
# decode step stops being free. Below it, extra sequences travel on a weight
# read that you already perform.
#
# This is Part 1's roofline, in sequences instead of FLOP per byte. Measure
# your own value in part2_pad_freeSequences. Then change this line.
B_RIDGE = 32

batch_sizes = np.array([1,2,4,8,16,32,64,128,256])
print(f'{len(lengths)} requests, median {np.median(lengths):.0f} tokens, '
      f'p99 {np.percentile(lengths,99):.0f}')
print(f'B_ridge = {B_RIDGE}')

# Exercise 1: how much does each batch size waste?

A static batch of B occupies B slots for `max(lengths)` steps. Useful work is
`sum(lengths)`. The rest is a finished sequence holding a slot open.

In [ ]:
def waste(batch):
  return 1 - batch.sum() / (batch.max() * len(batch))

def mean_waste(batch_size, trials=500):
  return np.mean([waste(rng.choice(lengths, batch_size)) for _ in range(trials)])

waste_fractions = np.array([mean_waste(int(batch_size)) for batch_size in batch_sizes])
for batch_size, waste_fraction in zip(batch_sizes, waste_fractions):
  print(f'batch {batch_size:>4}: {100*waste_fraction:5.1f}% wasted')

# Exercise 2: what you offered against what you delivered

Multiply the measured speedup by the fraction of slots that did real work,
and find the peak.

In [ ]:
offered = np.minimum(batch_sizes, B_RIDGE)          # min(B, B_ridge): the roofline

# what you actually get is what the hardware offered, minus the padding
delivered = offered * (1 - waste_fractions)

best = batch_sizes[np.argmax(delivered)]
print(f"{'batch':>6} {'offered':>8} {'wasted':>8} {'delivered':>10} {'kept':>6}")
for batch_size, offered_speedup, waste_fraction, delivered_speedup in zip(batch_sizes, offered, waste_fractions, delivered):
  mark = '  <-- best' if batch_size == best else ''
  print(f'{batch_size:>6} {offered_speedup:>7.0f}x {100*waste_fraction:>7.1f}% {delivered_speedup:>9.1f}x {100*delivered_speedup/offered_speedup:>5.0f}%{mark}')

# Exercise 3: can you cheat the distribution?

The tail sets the length of the batch. So keep the tail out of the batch:
gather a larger pool, sort it by length, and cut it into batches of similar
requests.

In [ ]:
def bucketed_waste(batch_size, num_buckets, trials=500):
  bucket_wastes = []
  for _ in range(trials):
    sampled_lengths = rng.choice(lengths, batch_size*num_buckets)
    sampled_lengths.sort()                                  # group similar lengths
    bucket_wastes.append(np.mean([waste(sampled_lengths[best_index*batch_size:(best_index+1)*batch_size]) for best_index in range(num_buckets)]))
  return np.mean(bucket_wastes)

batch_size = int(best)
print(f'batch {batch_size}, unsorted:            {100*mean_waste(batch_size):5.1f}% wasted')
for num_buckets in (2, 4, 8):
  print(f'batch {batch_size}, sorted into {num_buckets:>2} buckets: {100*bucketed_waste(batch_size, num_buckets):5.1f}% wasted')

# Exercise 4: change the machine, leave the traffic alone

Everything above used one value of `B_ridge`. Sweep it. For each machine,
report the best batch size and the fraction of the offered speedup that
survives padding.

In [ ]:
print(f"{'B_ridge':>8} {'best batch':>11} {'delivered':>10} {'kept':>6}")
for b_ridge in (8, 16, 32, 64, 128, 256):
  delivered_speedup = np.minimum(batch_sizes, b_ridge) * (1 - waste_fractions)
  best_index = int(np.argmax(delivered_speedup))
  print(f'{b_ridge:>8} {batch_sizes[best_index]:>11} {delivered_speedup[best_index]:>9.1f}x {100*delivered_speedup[best_index]/b_ridge:>5.0f}%')

### What survives a change of hardware

**The best batch size is `B_ridge`.** Do not memorise a number. It is the batch
at which your card stops giving sequences away free. It is the only property
of the machine in this notebook.

**And the fraction you keep falls as `B_ridge` rises.** A faster card permits a
larger batch. A larger batch holds a long outlier more often. The outlier sets
the length of the batch. So padding takes a larger share of a better machine.
That is the opposite of what you want.

So carry away a formula and a direction, not a measurement:

> delivered = `min(B, B_ridge)` x `(1 - waste(B))`

The first factor is your hardware. The second is your traffic. Better hardware
gives less and less until you fix the second factor.

**A sort by length helps a lot, and you cannot do it.** `sampled_lengths.sort()` used the
output length. That is the number of tokens that the model has not made yet.
You do know the *prompt* length, and you could sort by that. But prompt length
hardly predicts output length.

So one obvious fix needs an oracle, and the other uses a weak substitute. The
fix that works needs neither. It stops the engine from forming batches in
advance.

    ./vc guide 5